# Ingestion Exploration — Phase 2

**Purpose:** Inspect raw text extracted by `app/ingestion/parser.py` on the real Fall 2026 Prospectus PDF, *before* writing the cleaning logic.

We are specifically looking for:
- Repeated headers / footers (e.g. document title, page numbers) that appear on every page
- Broken line breaks (words split across lines)
- Extra whitespace / weird characters

This notebook does **not** modify any pipeline code — it's a read-only inspection step. Findings here will directly inform what `app/ingestion/cleaner.py` needs to handle.

In [11]:
import sys
from pathlib import Path

# Make `app` importable when running this notebook from the notebooks/ folder
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.ingestion.parser import extract_pdf_pages, summarize_extraction

PDF_PATH = project_root / "data" / "raw_documents" / "Prospectus - FALL 2026 (29-07-2026).pdf"

pages = extract_pdf_pages(PDF_PATH)
summary = summarize_extraction(pages)

print(f"Loaded: {PDF_PATH.name}")
print(f"Total pages: {summary['total_pages']}")
print(f"Empty pages: {summary['empty_pages']}")
print(f"Low-text pages: {summary['low_text_pages']}")
print(f"Avg chars/page: {summary['avg_chars_per_page']}")

Loaded: Prospectus - FALL 2026 (29-07-2026).pdf
Total pages: 382
Empty pages: 0
Low-text pages: 0
Avg chars/page: 1629.0


## 1. Raw text sample — first, middle, and last page

Sampling across the document (not just page 1) because prospectuses often have a very different layout on cover pages vs. body pages vs. index/appendix pages.

In [12]:
def show_page(page: dict, max_chars: int = 1200) -> None:
    text = page["text"]
    truncated = text[:max_chars]
    suffix = "\n... [truncated]" if len(text) > max_chars else ""

    print(f"{'=' * 70}")
    print(f"PAGE {page['page_number']}  |  char_count = {page['char_count']}")
    print(f"{'=' * 70}")
    print(truncated + suffix)
    print()


sample_indices = [0, len(pages) // 2, len(pages) - 1]  # first, middle, last

for idx in sample_indices:
    show_page(pages[idx])

PAGE 1  |  char_count = 72
UNIVERSITY OF
EDUCATION, LAHORE
FALL
www.ue.edu.pk
PROSPECTUS
UNIVERSITY

PAGE 192  |  char_count = 2514
software systems that solve challenging programming jobs. Computer Science 
spans the range from theory to models, design and programming. Computer 
Science offers a comprehensive foundation that permits graduates to adapt to 
new technologies and new ideas.
Program Vision
The BSCS (Post ADP) program aspires to innovate, broaden, publish and impart the 
advanced knowledge of computer science enabling students to participate and 
contribute in their field locally, nationally and globally through academia, research and 
applications.
Program Mission
The mission of the program is to impart modern, quality, comprehensive and effective 
theoretical as well as applied education in various domains of Computer Sciences. Also, to 
instill high degree professionalism in student by developing their communication, 
problem solving and technical skills to meet modern a

## 2. Header / footer pattern detection

For a set of pages, print just the **first 2 lines** and **last 2 lines** of each. If the same (or near-identical) text repeats across many pages, that is almost certainly a running header or footer that `cleaner.py` should strip out.

In [13]:
import random

random.seed(42)  # reproducible sample
sample_size = 10
sampled_pages = random.sample(pages, k=min(sample_size, len(pages)))
sampled_pages.sort(key=lambda p: p["page_number"])

print(f"{'PAGE':<6} | {'FIRST 2 LINES':<50} | {'LAST 2 LINES'}")
print("-" * 110)

for page in sampled_pages:
    lines = [l.strip() for l in page["text"].splitlines() if l.strip()]
    first_two = " | ".join(lines[:2]) if lines else "(empty)"
    last_two = " | ".join(lines[-2:]) if lines else "(empty)"
    print(f"{page['page_number']:<6} | {first_two[:50]:<50} | {last_two[:50]}")

PAGE   | FIRST 2 LINES                                      | LAST 2 LINES
--------------------------------------------------------------------------------------------------------------
13     | UNIVERSITY OF EDUCATION, LAHORE | FALL             | Encourage active student participation in sports a
53     | Mr. Muhammad Tehseen | Lecturer                    | (On Leave) | 47
58     | Dr. Sadia Shoukat | Associate Professor            | Lecturer | English
72     | UNIVERSITY OF EDUCATION, LAHORE | FALL             | ( - - - / Eve) | ( - - - / Eve)
115    | Ÿ | Select  and  use  appropriate  resource  mater | FALL | 109
126    | UNIVERSITY OF EDUCATION, LAHORE | FALL             | Students in this study area are tasked with conduc
141    | Ÿ | Application of economic theories and enhancing | FALL | 135
328    | UNIVERSITY OF EDUCATION, LAHORE | FALL             | universities), in tradisciplinary admissions may o
378    | UNIVERSITY OF EDUCATION, LAHORE | FALL             | For BS/B.Ed./B

## 3. Findings (fill in after reviewing output above)

- [ ] Repeated header text: 5
- [ ] Repeated footer text: 0
- [ ] Broken line breaks observed? No
- [ ] Any weird characters / encoding issues? No

These findings will define the exact rules implemented in `app/ingestion/cleaner.py`.